# How to Send Images to LLM

You can send images using a direct URL or a Base64 string. We also provide helper function to load a local image to base64.
- When using URL, the image image format is automatically guessed.
- When using Base64, you need specify the image format if it's different from default `jpeg`.
- Use `from_path` to load from local images.

In [1]:
import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images

## Example 1. Sending LLM image from URL

In [2]:
@kbench.task("Describe Image (URL)")
def describe_image_url(llm):
    """Sends an image URL directly to the model."""
    # Kaggle logo
    image_url = "https://www.kaggle.com/static/images/site-logo.png"

    # Create Image object from URL.
    # It will guess image type from URL.
    image = images.from_url(image_url)

    response = llm.prompt("What does this logo say?", image=image)

    kbench.assertions.assert_contains_regex(
        r"(?i)kaggle",
        response,
        expectation="LLM should identify the Kaggle logo.",
    )

describe_image_url.run(kbench.llm)

BokehModel(combine_events=True, render_bundle={'docs_json': {'0092c1bc-a5ad-42e7-a8b1-cf4dd323de84': {'version…

## Example 2. Sending LLM image from Base64 (specifying `format` parameter)

In [3]:
@kbench.task("Describe Image (Base64)")
def describe_image_base64(llm):
    """Sends a base64 encoded image with explicit format specification."""
    # Example: A small red dot (PNG)
    # This is a 1x1 red pixel in PNG format
    red_dot_b64 = (
        "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
    )

    # Create Image object from Base64, specifying the format as 'png'
    # The 'format' parameter is important when the image is not a JPEG (default)
    image = images.from_base64(red_dot_b64, format="png")

    response = llm.prompt("What color is this image?", image=image)

    kbench.assertions.assert_contains_regex(
        r"(?i)red",
        response,
        expectation="LLM should identify the color red.",
    )

describe_image_base64.run(kbench.llm)

BokehModel(combine_events=True, render_bundle={'docs_json': {'c617902e-2a89-4620-8f26-09e602b380b9': {'version…

## Example 3. Sending LLM image from local image file
You can use images from local files, e.g., by attaching a Kaggle dataset.

In [4]:
@kbench.task("Describe Image (Local File)")
def describe_local_image(llm):
    """Sends a local image with explicit format specification."""

    # Load a local image from an attached Kaggle dataset
    image = images.from_path("/kaggle/input/cross-modal-images/3_animals_cross_modal.jpeg")
    prompt = (
        "You are being provided with an image that has a number of objects pictured, "
        "and each has some associated text. You will be provided with a word, and you "
        "must respond with the name of the pictured object that is associated with the "
        "provided word.\nThe word is 'lion'. "
        "Do not provide thoughts, reasoning, adjectives or description. Respond with only a single word answer."
    )

    response = llm.prompt(prompt, image=image)

    kbench.assertions.assert_contains_regex(
        r"(?i)cow",
        response,
        expectation="LLM should associate the right animal.",
    )

describe_local_image.run(kbench.llm)

BokehModel(combine_events=True, render_bundle={'docs_json': {'32da62b1-bcfa-40c7-a60d-8c0ebe6753f6': {'version…

## Example 4. Comparing two images

In [5]:
@kbench.task("Compare Logos")
def compare_logos(llm):
    """Sends two images and asks for differences."""

    img1 = images.from_url(
        "https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png",
        caption="Logo 1",
    )
    img2 = images.from_url(
        "https://www.kaggle.com/static/images/logos/kaggle-logo-gray-300.png",
        caption="Logo 2",
    )

    # Use `send` to enable multi-image conversation.
    # Since `send` doesn't auto-convert URLs, we explicitly encode images to base64
    # so it will work with more models that don't directly accept an image URL.
    kbench.user.send(images.from_image_url(img1))
    kbench.user.send(images.from_image_url(img2))

    # For models that directly work with image URL
    # You can simply call
    # kbench.user.send(img1)
    # kbench.user.send(img2)

    response = llm.prompt("What are the main differences of these two images.")

    assessment = kbench.assertions.assess_response_with_judge(
        response_text=response,
        judge_llm=kbench.judge_llm,
        criteria=[
            "The answer should highlight the main difference is the background.",
            "The answer should mention the font are the same",
        ],
    )

    for result in assessment.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=f"Judge Criterion '{result.criterion}' should pass: {result.reason}",
        )


compare_logos.run(kbench.llm)

BokehModel(combine_events=True, render_bundle={'docs_json': {'3c624892-d17c-43ee-9029-d7c730220988': {'version…